<a href="https://colab.research.google.com/github/asmaatefomran/hirewise/blob/main/hirewise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Career Knowledge Assistant
### A RAG (Retrieval-Augmented Generation) Assistant with Conversational Memory

This notebook builds an AI assistant that answers questions about a CV (PDF) and a job-application tracking sheet (Google Sheet / CSV), using:

- **LangChain** — orchestration framework
- **HuggingFace embeddings** — turns text into vectors
- **FAISS** — vector database for semantic search
- **Groq (Llama 3.3 70B)** — the LLM that generates answers, free API
- **ConversationBufferMemory** — lets the assistant remember earlier turns


## 0. Setup

In [2]:
%pip install langchain==0.3.7 langchain-community==0.3.7 langchain-huggingface==0.1.2 langchain-groq==0.2.1 faiss-cpu==1.9.0 sentence-transformers==3.3.1 pypdf==5.1.0 pandas==2.2.3 python-dotenv==1.0.1 -q


In [6]:
%pip install --force-reinstall "numpy==1.26.4" "pandas==2.2.2" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2;

In [1]:
import os
from pathlib import Path
import pandas as pd

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate
from google.colab import userdata; import os; os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

DATA_DIR = Path("data")
INDEX_DIR = Path("faiss_index")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Imports OK")

Imports OK


In [2]:
assert os.getenv("GROQ_API_KEY"), (
    "GROQ_API_KEY not found. Create a .env file in this folder with "
    "GROQ_API_KEY=your_key_here (see .env.example)."
)
print("Groq API key loaded successfully")

Groq API key loaded successfully


## 1. Data Integration

Load content from:
- **A PDF** (the CV) — unstructured source
- **A Google Sheet**, represented here as a CSV export (the job-application tracker) — structured source

Each becomes a LangChain `Document`. For the sheet, every row is converted into a short natural-language paragraph so it embeds and retrieves the same way as prose text.

In [3]:
def load_pdf_documents(pdf_path: Path) -> list[Document]:
    """Load a PDF (e.g. CV) into LangChain Documents, one per page."""
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = pdf_path.name
        d.metadata["source_type"] = "pdf"
    return docs


def load_sheet_documents(csv_path: Path) -> list[Document]:
    """
    Load a 'Google Sheet' (here a CSV export) and convert each row into a
    natural-language Document.

    To pull live from Google Sheets instead of a CSV export, swap this
    function's body for something like:

        import gspread
        gc = gspread.service_account(filename="credentials.json")
        sheet = gc.open("Job Applications").sheet1
        rows = sheet.get_all_records()
    """
    df = pd.read_csv(csv_path)
    docs = []
    for i, row in df.iterrows():
        text = (
            f"Company: {row['company']}\n"
            f"Role: {row['role']}\n"
            f"Status: {row['status']}\n"
            f"Date applied: {row['date_applied']}\n"
            f"Contact: {row['contact']}\n"
            f"Notes: {row['notes']}"
        )
        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": csv_path.name,
                    "source_type": "sheet",
                    "row": i,
                    "company": row["company"],
                    "status": row["status"],
                },
            )
        )
    return docs


pdf_docs = load_pdf_documents(DATA_DIR / "cv.pdf")
sheet_docs = load_sheet_documents(DATA_DIR / "companies_tracker.csv")

print(f"PDF pages loaded: {len(pdf_docs)}")
print(f"Sheet rows loaded: {len(sheet_docs)}")
print()
print("Sample sheet document:")
print(sheet_docs[0].page_content)

PDF pages loaded: 1
Sheet rows loaded: 9

Sample sheet document:
Company: Instabug
Role: Backend Engineer
Status: Applied
Date applied: 2026-06-02
Contact: Referral via graduation project defense contact
Notes: Waiting on recruiter reply; role focuses on Node.js and monitoring SDKs


## 2. Embedding & Indexing

Split the documents into small overlapping chunks, convert each chunk into a vector embedding using a HuggingFace sentence-transformer model, and store all vectors in a local FAISS index for fast semantic search.

In [4]:
all_docs = pdf_docs + sheet_docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(all_docs)
print(f"Total chunks: {len(chunks)}")

Total chunks: 14


In [5]:
print("Downloading/loading embedding model (first run takes a minute)...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)

INDEX_DIR.mkdir(exist_ok=True)
vectorstore.save_local(str(INDEX_DIR))
print(f"FAISS index saved to {INDEX_DIR}/")

Downloading/loading embedding model (first run takes a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index...
FAISS index saved to faiss_index/


## 3. Question-Answering Pipeline (RAG)

Wire the FAISS index up as a **retriever**, connect it to an LLM (Groq's free Llama 3.3 70B), and combine them into a Retrieval-Augmented Generation chain: the LLM only answers using the chunks retrieved from the index, so it stays grounded in your actual data instead of guessing.

In [6]:
CUSTOM_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a career knowledge assistant. Answer the user's question "
        "using ONLY the context below, which comes from their CV and their "
        "job-application tracking sheet. If the answer isn't in the context, "
        "say you don't have that information instead of guessing.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("Retriever + LLM ready")

Retriever + LLM ready


## 4. Memory Integration

Add `ConversationBufferMemory`, which stores the full chat history. This is what lets the assistant resolve follow-up questions like *"who's my contact **there**?"* — where "there" only makes sense in light of the previous turn.

In [7]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": CUSTOM_PROMPT},
)

print("Conversational RAG chain ready")

Conversational RAG chain ready


/tmp/ipykernel_3748/2062077593.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [8]:
def ask(question: str) -> dict:
    result = chain.invoke({"question": question})
    return {
        "answer": result["answer"],
        "sources": [
            {"source": d.metadata.get("source"), "type": d.metadata.get("source_type")}
            for d in result.get("source_documents", [])
        ],
    }

### Try it — single question

In [9]:
result = ask("What did I do during my Sahl internship?")
print("Answer:", result["answer"])
print("Sources:", result["sources"])

Answer: During your Sahl internship, you worked on backend services using .NET and C#, and contributed to a Flutter mobile client. You also set up something, but the details of what you set up are not specified in the context.
Sources: [{'source': 'companies_tracker.csv', 'type': 'sheet'}, {'source': 'companies_tracker.csv', 'type': 'sheet'}, {'source': 'cv.pdf', 'type': 'pdf'}, {'source': 'companies_tracker.csv', 'type': 'sheet'}]


### Try it — memory / follow-up questions

This proves memory is actually working: "there" and "its" only resolve correctly because the chain remembers the previous turn.

In [10]:
r1 = ask("Tell me about my application to Instabug.")
print("Q1:", r1["answer"], "\n")

r2 = ask("Who's my contact there?")
print("Q2 (follow-up):", r2["answer"], "\n")

r3 = ask("And what's its current status?")
print("Q3 (follow-up):", r3["answer"])

Q1: The status of your application to Instabug is "Applied" and the details are as follows: you applied on 2026-06-02 through a referral via your graduation project defense contact. The role you applied for is Backend Engineer, which focuses on Node.js and monitoring SDKs, and you are currently waiting on a reply from the recruiter. 

Q2 (follow-up): Your contact at Instabug is a referral via your graduation project defense. 

Q3 (follow-up): The current status of your application to Instabug is "Applied", and you are waiting on a recruiter reply.


### Interactive chat loop (optional)

Run this cell to chat freely. Type `exit` to stop.

In [11]:
while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit"}:
        break
    result = ask(q)
    print(f"\nAssistant: {result['answer']}")
    srcs = ", ".join(f"{s['source']} ({s['type']})" for s in result["sources"])
    print(f"[sources: {srcs}]\n")

You: exit


## 5. Testing & Evaluation

Run a fixed set of test questions and check whether expected keywords appear in the answer — a simple, transparent way to measure retrieval relevance. Also runs a dedicated multi-turn sequence to specifically verify memory/follow-up handling.



In [12]:
SINGLE_TURN_TESTS = [
    ("What did I do during my Sahl internship?", ["net", "c#", "flutter", "ci/cd", "fintech"]),
    ("What tech stack did I use at Genesis Creations?", ["node", "express", "mongodb"]),
    ("Which companies haven't I heard back from?", ["careem", "breadfast"]),
    ("What was my graduation project about?", ["defect", "random forest", "jm1", "defexhunter"]),
    ("Am I still waiting on a technical interview anywhere?", ["noon"]),
    ("What was the outcome at Vezeeta?", ["reject"]),
]

def build_fresh_chain():
    fresh_memory = ConversationBufferMemory(
        memory_key="chat_history", return_messages=True, output_key="answer",
    )
    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=fresh_memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": CUSTOM_PROMPT},
    )

passed = 0
for question, keywords in SINGLE_TURN_TESTS:
    test_chain = build_fresh_chain()
    result = test_chain.invoke({"question": question})
    answer_lower = result["answer"].lower()
    hit = any(kw.lower() in answer_lower for kw in keywords)
    passed += hit
    status = "PASS" if hit else "FAIL"
    print(f"[{status}] Q: {question}")
    print(f"       A: {result['answer']}\n")

print(f"Single-turn relevance score: {passed}/{len(SINGLE_TURN_TESTS)}")

[PASS] Q: What did I do during my Sahl internship?
       A: During your Sahl internship, you worked on backend services using .NET and C#, and contributed to a Flutter mobile client. You also set up something, but the details of what you set up are not specified in the context.

[PASS] Q: What tech stack did I use at Genesis Creations?
       A: At Genesis Creations, you used Node.js, Express.js, and MongoDB as the primary data store to build and maintain REST APIs.

[PASS] Q: Which companies haven't I heard back from?
       A: You haven't heard back from Breadfast.

[FAIL] Q: What was my graduation project about?
       A: I don't have that information. The context provided does not mention a graduation project. It only mentions the degree and GPA, but does not provide details about a specific project.

[PASS] Q: Am I still waiting on a technical interview anywhere?
       A: Yes, you are still waiting on a technical interview at Datenlotsen (octo education GmbH) and also waiting to

In [13]:
MEMORY_TEST_SEQUENCE = [
    ("Tell me about my application to Instabug.", None),
    ("Who's my contact there?", ["referral", "graduation project"]),
    ("And what's its current status?", ["applied", "waiting"]),
]

mem_chain = build_fresh_chain()
mem_passed, mem_total = 0, 0
for question, keywords in MEMORY_TEST_SEQUENCE:
    result = mem_chain.invoke({"question": question})
    print(f"Q: {question}")
    print(f"A: {result['answer']}\n")
    if keywords:
        mem_total += 1
        hit = any(kw.lower() in result["answer"].lower() for kw in keywords)
        mem_passed += hit
        print(f"  [{'PASS' if hit else 'FAIL'}] expected one of: {keywords}\n")

print(f"Memory/follow-up score: {mem_passed}/{mem_total}")

Q: Tell me about my application to Instabug.
A: You applied to Instabug for the Backend Engineer role on 2026-06-02. The application was made through a referral contact from your graduation project defense. The role you applied for focuses on Node.js and monitoring SDKs. Your current status is "Applied" and you are waiting to hear back from the recruiter.

Q: Who's my contact there?
A: Your referral contact at Instabug is from your graduation project defense.

  [PASS] expected one of: ['referral', 'graduation project']

Q: And what's its current status?
A: The current status of your application to Instabug for the Backend Engineer role is "Applied", with a note that you are waiting on a recruiter reply.

  [PASS] expected one of: ['applied', 'waiting']

Memory/follow-up score: 2/2


In [14]:
total_passed = passed + mem_passed
total_all = len(SINGLE_TURN_TESTS) + mem_total
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Single-turn relevance: {passed}/{len(SINGLE_TURN_TESTS)}")
print(f"Memory/follow-up:      {mem_passed}/{mem_total}")
print(f"Overall:               {total_passed}/{total_all} ({100*total_passed/total_all:.0f}%)")

SUMMARY
Single-turn relevance: 5/6
Memory/follow-up:      2/2
Overall:               7/8 (88%)
